# Customer Shopping Behavior Analysis

**Portfolio notebook:** data quality, cleaning, feature engineering, EDA, KPI analysis, and business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path('../data/raw/customer_shopping_behavior.csv')
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all').T)
print('Duplicates:', df.duplicated().sum())
print('Missing values:
', df.isna().sum().sort_values(ascending=False))

## Data Cleaning
Review ratings are missing for a small number of records. Because rating distributions can differ by category, impute missing ratings with the category median rather than the global median.

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))
df.columns = df.columns.str.lower().str.replace(' ', '_', regex=False)
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

frequency_mapping = {'Fortnightly':14,'Weekly':7,'Monthly':30,'Quarterly':90,'Bi-Weekly':14,'Annually':365,'Every 3 Months':90}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df['age_group'] = pd.cut(df['age'], [0,19,29,39,49,59,200], labels=['Teen','20-29','30-39','40-49','50-59','60+'], include_lowest=True)
df['customer_segment'] = pd.cut(df['previous_purchases'], [0,1,10,np.inf], labels=['New','Returning','Loyal'], include_lowest=True)
df['discount_flag'] = df['discount_applied'].eq('Yes').astype(int)
df['subscription_flag'] = df['subscription_status'].eq('Yes').astype(int)
df.isna().sum().sort_values(ascending=False).head()

## KPI Summary

In [ ]:
kpis = pd.Series({
    'Total Customers': df['customer_id'].nunique(),
    'Total Revenue': df['purchase_amount'].sum(),
    'Average Purchase': df['purchase_amount'].mean(),
    'Average Rating': df['review_rating'].mean(),
    'Subscription Rate': df['subscription_flag'].mean(),
    'Discount Usage Rate': df['discount_flag'].mean()
})
display(kpis)

## Revenue and Product Analysis

In [ ]:
category_summary = df.groupby('category').agg(orders=('customer_id','count'), revenue=('purchase_amount','sum'), avg_purchase=('purchase_amount','mean'), avg_rating=('review_rating','mean')).sort_values('revenue', ascending=False)
display(category_summary.round(2))

plt.figure(figsize=(9,5))
sns.barplot(data=category_summary.reset_index(), x='revenue', y='category')
plt.title('Revenue by Category')
plt.xlabel('Revenue (USD)'); plt.ylabel('Category'); plt.tight_layout(); plt.show()

## Subscription Analysis

In [ ]:
subscription_summary = df.groupby('subscription_status').agg(customers=('customer_id','nunique'), revenue=('purchase_amount','sum'), avg_purchase=('purchase_amount','mean')).sort_values('revenue', ascending=False)
display(subscription_summary.round(2))

## Customer Segmentation

In [ ]:
segment_summary = df.groupby('customer_segment', observed=True).agg(customers=('customer_id','nunique'), revenue=('purchase_amount','sum'), avg_purchase=('purchase_amount','mean')).sort_values('revenue', ascending=False)
display(segment_summary.round(2))

## Discount Analysis

In [ ]:
discount_summary = df.groupby('discount_applied').agg(orders=('customer_id','count'), revenue=('purchase_amount','sum'), avg_purchase=('purchase_amount','mean'))
display(discount_summary.round(2))

## Product Performance

In [ ]:
product_summary = df.groupby('item_purchased').agg(orders=('customer_id','count'), revenue=('purchase_amount','sum'), avg_rating=('review_rating','mean')).sort_values('revenue', ascending=False)
display(product_summary.head(10).round(2))

## Findings
Use the generated tables above to document the top revenue categories, customer segments, subscription differences, discount behavior, and products. Avoid causal language unless supported by an appropriate causal design.